In [1]:
"""
MF Monthly Returns — Filter & Sample from Existing CSVs
=========================================================
Uses already-downloaded data from v2 script. No scraping.

Inputs  (edit paths below if needed):
  - mf_monthly_returns_long.csv   ← from v2 script
  - scheme_list.csv               ← from v2 script

Outputs:
  - mf_filtered_wide.csv          ← ~200-250 funds, months as rows
  - mf_filtered_long.csv          ← same data, long format with Category + AMC columns

Run:
    python mf_filter_existing.py
"""

import pandas as pd
import logging

# ─────────────────────────────────────────
# CONFIG — update these paths if needed
# ─────────────────────────────────────────
INPUT_WIDE        = "/content/mf_monthly_returns_wide.csv"
OUTPUT_WIDE       = "mf_filtered_wide.csv"
OUTPUT_LONG       = "mf_filtered_long.csv"

MIN_MONTHS        = 60    # fund must have at least this many months of data
MAX_PER_CATEGORY  = 30    # max funds sampled per category
RANDOM_SEED       = 42

# ─────────────────────────────────────────
# LOGGING
# ─────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger(__name__)

# ─────────────────────────────────────────
# SEBI CATEGORY CLASSIFICATION
# ─────────────────────────────────────────
CATEGORIES = {
    "Large Cap":           ["large cap", "largecap", "large-cap", "bluechip", "blue chip"],
    "Mid Cap":             ["mid cap", "midcap", "mid-cap"],
    "Small Cap":           ["small cap", "smallcap", "small-cap"],
    "Flexi/Multi Cap":     ["flexi cap", "flexicap", "flexi-cap", "multi cap", "multicap", "multi-cap"],
    "ELSS":                ["elss", "tax saver", "taxsaver", "tax saving"],
    "Aggressive Hybrid":   ["aggressive hybrid", "equity hybrid", "balanced advantage", "dynamic asset"],
    "Conservative Hybrid": ["conservative hybrid", "equity savings", "arbitrage"],
    "Short Duration Debt": ["short duration", "short term", "low duration", "ultra short",
                            "money market", "liquid", "overnight"],
    "Medium Duration Debt":["medium duration", "medium term", "corporate bond",
                            "banking and psu", "banking & psu"],
    "Long Duration Debt":  ["long duration", "long term", "gilt", "10 year"],
    "Index / ETF":         ["index fund", "nifty 50", "sensex", "nifty next 50", "nifty 100"],
    "Sectoral":            ["pharma", "technology", "tech", "banking", "infra", "infrastructure",
                            "consumption", "energy", "fmcg", "healthcare", "auto"],
    "International":       ["international", "global", "overseas", "us equity", "nasdaq", "s&p"],
    "Thematic":            ["esg", "dividend yield", "value fund", "contra", "focused fund"],
}

AMC_PREFIXES = [
    "aditya birla sun life", "axis", "bandhan", "baroda bnp paribas", "bnp paribas",
    "canara robeco", "dsp", "edelweiss", "franklin", "hdfc", "hsbc", "icici prudential",
    "idbi", "idfc first", "invesco", "iti", "jm financial", "kotak", "lic",
    "mahindra manulife", "mirae asset", "motilal oswal", "navi", "nippon india",
    "nj", "old bridge", "parag parikh", "pgim india", "ppfas", "quant",
    "quantum", "sbi", "shriram", "sundaram", "tata", "taurus", "union",
    "uti", "whiteoak", "zerodha"
]


def classify_fund(name: str) -> str:
    name_lower = name.lower()
    for category, keywords in CATEGORIES.items():
        if any(kw in name_lower for kw in keywords):
            return category
    return "Other"


def extract_amc(name: str) -> str:
    name_lower = name.lower()
    for prefix in AMC_PREFIXES:
        if name_lower.startswith(prefix):
            return prefix.title()
    return name.split()[0]


def is_direct_growth(name: str) -> bool:
    name_lower = name.lower()
    has_direct = "direct" in name_lower
    has_growth = (
        "growth" in name_lower and
        "dividend" not in name_lower and
        "idcw" not in name_lower and
        "payout" not in name_lower and
        "reinvest" not in name_lower
    )
    return has_direct and has_growth


# ─────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────
def main():
    log.info("=" * 60)
    log.info("MF Filter & Sample (no scraping)")
    log.info("=" * 60)

    # ── Load existing wide data ──
    log.info(f"Loading {INPUT_WIDE} ...")
    df_wide = pd.read_csv(INPUT_WIDE, index_col=0)
    log.info(f"  Loaded {df_wide.shape[0]} months × {df_wide.shape[1]:,} funds")

    all_fund_names = df_wide.columns.tolist()

    # ── Step 1: Direct + Growth filter ──
    direct_growth = [f for f in all_fund_names if is_direct_growth(f)]
    log.info(f"After Direct+Growth filter: {len(direct_growth):,} funds")

    # ── Step 2: Minimum months filter ──
    valid_funds = [
        f for f in direct_growth
        if df_wide[f].notna().sum() >= MIN_MONTHS
    ]
    log.info(f"After min {MIN_MONTHS} months filter: {len(valid_funds):,} funds")

    # ── Step 3: Classify each fund ──
    fund_meta = pd.DataFrame({
        "Fund Name": valid_funds,
        "Category":  [classify_fund(n) for n in valid_funds],
        "AMC":       [extract_amc(n) for n in valid_funds],
    })

    log.info("\nFunds by category (before sampling):")
    for cat, cnt in fund_meta["Category"].value_counts().items():
        log.info(f"  {cat:30s}: {cnt}")

    # ── Step 4: Stratified sample — one per AMC per category, capped ──
    sampled_names = []
    for cat, group in fund_meta.groupby("Category"):
        one_per_amc = group.drop_duplicates(subset="AMC")
        if len(one_per_amc) <= MAX_PER_CATEGORY:
            picks = one_per_amc
        else:
            picks = one_per_amc.sample(MAX_PER_CATEGORY, random_state=RANDOM_SEED)
        sampled_names.extend(picks["Fund Name"].tolist())
        log.info(f"  Sampled {len(picks):3d} funds from {cat}")

    log.info(f"\nFinal sample: {len(sampled_names)} funds")

    # ── Filter wide df to sampled funds ──
    df_wide = df_wide[sampled_names].sort_index().round(4)
    df_wide.index.name = "Month"
    df_wide.to_csv(OUTPUT_WIDE)
    log.info(f"Wide CSV → {OUTPUT_WIDE}  [{df_wide.shape[0]} months × {df_wide.shape[1]} funds]")

    # ── Melt to long format with Category + AMC ──
    meta_map = fund_meta.set_index("Fund Name")[["Category", "AMC"]].to_dict("index")
    df_long = (
        df_wide
        .reset_index()
        .melt(id_vars="Month", var_name="Fund Name", value_name="Monthly Return (%)")
        .dropna(subset=["Monthly Return (%)"])
    )
    df_long["Category"] = df_long["Fund Name"].map(lambda x: meta_map[x]["Category"])
    df_long["AMC"]      = df_long["Fund Name"].map(lambda x: meta_map[x]["AMC"])
    df_long = df_long[["Month", "Fund Name", "AMC", "Category", "Monthly Return (%)"]]
    df_long = df_long.sort_values(["Fund Name", "Month"]).reset_index(drop=True)
    df_long.to_csv(OUTPUT_LONG, index=False)
    log.info(f"Long CSV → {OUTPUT_LONG}  [{len(df_long):,} rows]")

    # ── Final summary ──
    print("\n" + "=" * 55)
    print("SUMMARY")
    print("=" * 55)
    print(f"  Months : {df_wide.index[0]} → {df_wide.index[-1]}")
    print(f"  Funds  : {df_wide.shape[1]}")
    print(f"  Rows (long) : {len(df_long):,}")
    print(f"  Avg fill rate: {df_wide.notna().mean().mean()*100:.1f}%")
    print()
    print("  Final funds by category:")
    cat_summary = df_long.drop_duplicates("Fund Name")["Category"].value_counts()
    for cat, cnt in cat_summary.items():
        print(f"    {cat:30s}: {cnt}")
    print()
    print("  Files saved:")
    print(f"    {OUTPUT_WIDE}")
    print(f"    {OUTPUT_LONG}")


if __name__ == "__main__":
    main()


SUMMARY
  Months : 2020-02 → 2026-03
  Funds  : 360
  Rows (long) : 26,428
  Avg fill rate: 99.2%

  Final funds by category:
    Other                         : 30
    Short Duration Debt           : 30
    ELSS                          : 30
    Flexi/Multi Cap               : 30
    Aggressive Hybrid             : 30
    Mid Cap                       : 29
    Conservative Hybrid           : 26
    Large Cap                     : 26
    Thematic                      : 23
    Sectoral                      : 22
    Medium Duration Debt          : 20
    Small Cap                     : 20
    Long Duration Debt            : 18
    Index / ETF                   : 15
    International                 : 11

  Files saved:
    mf_filtered_wide.csv
    mf_filtered_long.csv
